# Tin tức (Unstructured Data) — One-layer Ingestion

Notebook điều khiển luồng ingestion 1 tầng cho 3 nguồn `vnstock`, `rss`, `html`.

Mỗi nguồn ghi output riêng theo layout (append-only):
- `news/<source>/date=<run_date>/PART-<run_id>.parquet`

Partition theo ngày (date=YYYY-MM-DD). Khi rerun cùng ngày có thể truncate partition rồi ghi file mới (không update in-place).

Schema của cả 3 output là thống nhất theo `NEWS_COLUMNS`, phục vụ downstream sentiment analysis.

In [1]:
import os, sys, warnings, threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

_orig_hook = threading.excepthook
def _quiet_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass
    else:
        _orig_hook(args)
threading.excepthook = _quiet_hook
warnings.filterwarnings("ignore")

from pathlib import Path
root = Path.cwd()
if not (root / "ingestion").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from ingestion.common import (
    configure_logging,
    load_dotenv_from_project_root,
    register_vnstock_api_key_from_env,
 )
from ingestion.unstructured_data import (
    NEWS_COLUMNS,
    NewsIngestionConfig,
    ingest_news,
    validate_news_df,
 )

configure_logging()
load_dotenv_from_project_root()
register_vnstock_api_key_from_env()
print("[OK] setup")

📦 **Vnstock 3.5.1 is available**
Current: 3.5.0 (Python 3.12 (venv))
Update: `C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\venv\Scripts\python.exe -m pip install vnstock --upgrade`
Release: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban

📦 **Vnai 2.4.6 is available**
Current: 2.4.0 (Python 3.12 (venv))
Update: `C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\venv\Scripts\python.exe -m pip install vnai --upgrade`
Release: https://pypi.org/project/vnai/#history

2026-04-18 00:18:49 [INFO] Đã nạp biến môi trường từ C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\.env
2026-04-18 00:18:49 [INFO] Đã nạp biến môi trường từ C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\.env
2026-04-18 00:18:50 [INFO] API key setup completed
2026-04-18 00:18:50 [INFO] Đã gọi register_user từ VNSTOCK_API_KEY.


✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! vnst***6437
✓ Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
[OK] setup


## Cấu hình

In [2]:
api_key = os.getenv("VNSTOCK_API_KEY", "").strip()
rate_limit = 60 if api_key else 20

news_cfg = NewsIngestionConfig(
    use_listing_tickers=True,
    listing_exchange_filter=["HSX", "HNX"],
    max_tickers_per_run=50,
    max_articles_per_source=200,
    days_back=0,
    days_back_vnstock=7,
    days_back_rss=1,
    days_back_html=1,
    rate_limit_rpm=rate_limit,
    enable_vnstock=True,
    enable_rss=True,
    enable_html=True,
    enable_ticker_match=True,
    append_only=True,
    truncate_partition=True,
 )

print(f"run_date     : {news_cfg.run_date}")
print(f"news_root    : {news_cfg.news_root}")
print(f"sources.yaml : {news_cfg.resolved_sources_yaml()}")
print(f"listing      : {news_cfg.resolved_listing_parquet()}")
print(f"listing_ok   : {news_cfg.resolved_listing_parquet().is_file()}")
print(f"tickers      : {len(news_cfg.resolved_tickers())}")
print(f"days_back    : vnstock={news_cfg.days_back_vnstock}d rss={news_cfg.days_back_rss}d html={news_cfg.days_back_html}d (fallback={news_cfg.days_back})")
print(f"rate_limit   : {news_cfg.rate_limit_rpm} rpm (api_key={'yes' if api_key else 'no'})")
print(f"append_only  : {news_cfg.append_only} (truncate_partition={news_cfg.truncate_partition})")
print(f"schema       : {NEWS_COLUMNS}")

run_date     : 2026-04-18
news_root    : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news
sources.yaml : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\ingestion\unstructured_data\sources.yaml
listing      : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\listing\master\listing.parquet
listing_ok   : True
tickers      : 50
days_back    : vnstock=7d rss=1d html=1d (fallback=0)
rate_limit   : 60 rpm (api_key=yes)
append_only  : True (truncate_partition=True)
schema       : ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']


## Chạy

In [3]:
news_paths = ingest_news(news_cfg)
news_paths

2026-04-18 00:18:53 [INFO] Đã nạp biến môi trường từ C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\.env
2026-04-18 00:18:53 [INFO] API key setup completed
2026-04-18 00:18:53 [INFO] Đã gọi register_user từ VNSTOCK_API_KEY.
2026-04-18 00:18:53 [INFO] vnstock news: 50 tickers, max_per=200


✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! vnst***6437
✓ Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)


2026-04-18 00:18:54 [INFO] vnstock news YEG@kbs: 1 rows
2026-04-18 00:18:55 [INFO] vnstock news YBM@kbs: 1 rows
2026-04-18 00:18:56 [INFO] vnstock news X20@kbs: 1 rows
2026-04-18 00:18:57 [INFO] vnstock news WSS@kbs: 1 rows
2026-04-18 00:18:57 [INFO] vnstock news WCS@kbs: 1 rows
2026-04-18 00:18:59 [INFO] vnstock news VVS@kbs: 1 rows
2026-04-18 00:19:00 [INFO] vnstock news VTZ@kbs: 1 rows
2026-04-18 00:19:01 [INFO] vnstock news VTV@kbs: 1 rows
2026-04-18 00:19:02 [INFO] vnstock news VTP@kbs: 1 rows
2026-04-18 00:19:02 [INFO] vnstock news VTO@kbs: 1 rows
2026-04-18 00:19:04 [INFO] vnstock news VTJ@kbs: 1 rows
2026-04-18 00:19:04 [INFO] vnstock news VTH@kbs: 1 rows
2026-04-18 00:19:06 [INFO] vnstock news VTC@kbs: 1 rows
2026-04-18 00:19:06 [INFO] vnstock news VTB@kbs: 1 rows
2026-04-18 00:19:08 [INFO] vnstock news VSM@kbs: 1 rows
2026-04-18 00:19:09 [INFO] vnstock news VSI@kbs: 1 rows
2026-04-18 00:19:10 [INFO] vnstock news VSH@kbs: 1 rows
2026-04-18 00:19:11 [INFO] vnstock news VSC@kbs:

{'row_counts': {'vnstock': 5, 'rss': 55, 'html': 100},
 'vnstock': {'parquet': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\vnstock\\date=2026-04-18\\PART-172233837571.parquet'},
 'rss': {'parquet': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\rss\\date=2026-04-18\\PART-172233887646.parquet'},
 'html': {'parquet': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\html\\date=2026-04-18\\PART-172233912239.parquet'}}

## Kiểm tra kết quả

In [4]:
import pandas as pd

print("=" * 70)
print("OUTPUT FILES")
print("=" * 70)
for key in ["vnstock", "rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    csv_path = info.get("csv", "")
    if not parquet_path:
        print(f"  {key:8s}: (trống)")
        continue
    df = pd.read_parquet(parquet_path)
    print(f"  {key:8s}: {len(df):>5d} dòng")
    print(f"    parquet: {parquet_path}")
    print(f"    csv    : {csv_path}")

print(f"\nRow counts: {news_paths.get('row_counts', {})}")

OUTPUT FILES
  vnstock :     5 dòng
    parquet: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\vnstock\date=2026-04-18\PART-172233837571.parquet
    csv    : 
  rss     :    55 dòng
    parquet: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\rss\date=2026-04-18\PART-172233887646.parquet
    csv    : 
  html    :   100 dòng
    parquet: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\html\date=2026-04-18\PART-172233912239.parquet
    csv    : 

Row counts: {'vnstock': 5, 'rss': 55, 'html': 100}


## Validation & Data Quality

In [5]:
# Validate per-source output with NEWS_COLUMNS
for key in ["vnstock", "rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        print(f"{key}: no output")
        continue

    df = pd.read_parquet(parquet_path)
    issues = validate_news_df(df)
    print(f"{key}: {len(df)} rows")
    if issues:
        print("  ⚠ issues:")
        for i in issues:
            print(f"    - {i}")
    else:
        print("  ✓ validation passed")
    print(f"  columns: {list(df.columns)}")

vnstock: 5 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']
rss: 55 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']
html: 100 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']


In [6]:
# Quick preview for sentiment-ready fields
for key in ["vnstock", "rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        continue
    df = pd.read_parquet(parquet_path)
    print(f"\n[{key}] preview")
    print(df[["article_id", "source", "ticker", "title", "body_text", "published_at"]].head(3))


[vnstock] preview
                                          article_id   source ticker  \
0  08d5d3a5c2a2f77a67d6e4b124b9e572567c904dac6d6b...  vnstock    VPX   
1  6fae0a593e80634cb712fdb7c6c4bad546dcaed04eb104...  vnstock    VPG   
2  85eadb28623ef1e90f97ea1a6f8382bf2f65bed6cffc8e...  vnstock    VPB   

                                               title body_text  \
0   Phó Chủ tịch VPBankS từ nhiệm trước thềm đại hội             
1  Lộ diện doanh nghiệp niêm yết lỗ nặng nhất năm...             
2  VPB: Link công bố cập nhật, bổ sung tài liệu Đ...             

                      published_at  
0 2026-03-30 10:10:32.193000+00:00  
1 2026-02-03 10:14:19.317000+00:00  
2                              NaT  

[rss] preview
                                          article_id         source ticker  \
0  253a8662673287a5ae9a6de9a8e55272602856739c58e5...  vnexpress_rss    NaN   
1  9f284a8bd7bc2f9b5cdbf4bb2af783c4a4d75fbccb953c...  vnexpress_rss    NaN   
2  c3ec59287fff904e830b24958f9

In [7]:
# Data size & quality audit for Unstructure_Data/news
from pathlib import Path
import pandas as pd

scan_days = 30  # số ngày gần nhất để kiểm tra
min_rows_per_day = {"vnstock": 100, "rss": 50, "html": 50}
sources = ["vnstock", "rss", "html"]

if "news_cfg" in globals() and hasattr(news_cfg, "news_root"):
    news_root = Path(news_cfg.news_root)
else:
    news_root = Path("data-lake/raw/Unstructure_Data/news")

def _partition_date(path: Path) -> pd.Timestamp:
    name = path.name
    if name.startswith("date="):
        return pd.to_datetime(name.split("=", 1)[1], errors="coerce")
    return pd.NaT

today = pd.Timestamp.today().normalize()
cutoff = today - pd.Timedelta(days=max(scan_days - 1, 0))

summary_rows = []
daily_rows = []

print(f"news_root: {news_root}")
print(f"scan_days: {scan_days} (cutoff: {cutoff.date()})")
print(f"min_rows_per_day: {min_rows_per_day}")

for source in sources:
    source_dir = news_root / source
    if not source_dir.exists():
        print(f"[{source}] folder not found: {source_dir}")
        continue
    date_dirs = sorted([p for p in source_dir.glob("date=*") if p.is_dir()])
    total_rows = 0
    total_files = 0
    days_with_data = 0
    min_rows = None
    max_rows = None
    min_pub = None
    max_pub = None
    required = ["article_id", "source", "title", "url", "fetched_at"]
    null_counts = {c: 0 for c in required}
    total_count = 0
    all_ids = set()
    total_ids = 0

    for date_dir in date_dirs:
        dt = _partition_date(date_dir)
        if pd.isna(dt) or dt < cutoff:
            continue
        files = sorted(date_dir.glob("*.parquet"))
        if not files:
            continue
        try:
            df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
        except Exception as ex:
            print(f"[{source}] read error at {date_dir}: {ex}")
            continue
        rows = len(df)
        if rows == 0:
            continue
        days_with_data += 1
        total_rows += rows
        total_files += len(files)
        min_rows = rows if min_rows is None else min(min_rows, rows)
        max_rows = rows if max_rows is None else max(max_rows, rows)

        # published_at range
        if "published_at" in df.columns:
            pub = pd.to_datetime(df["published_at"], errors="coerce", utc=True)
            if pub.notna().any():
                cur_min = pub.min()
                cur_max = pub.max()
                min_pub = cur_min if min_pub is None else min(min_pub, cur_min)
                max_pub = cur_max if max_pub is None else max(max_pub, cur_max)

        # required nulls
        total_count += rows
        for col in required:
            if col in df.columns:
                null_counts[col] += int(df[col].isna().sum())
            else:
                null_counts[col] += rows

        # dedup ratio
        if "article_id" in df.columns:
            ids = df["article_id"].dropna().astype(str)
            total_ids += len(ids)
            all_ids.update(ids)

        daily_rows.append({"source": source, "date": dt.date().isoformat(), "rows": rows})

    if days_with_data == 0:
        summary_rows.append({"source": source, "days_with_data": 0, "total_rows": 0})
        continue

    avg_rows = total_rows / days_with_data
    dup_rate = 0.0 if total_ids == 0 else 1 - (len(all_ids) / total_ids)
    null_rates = {c: (null_counts[c] / total_count) if total_count else 0 for c in required}
    ok_days = sum(1 for d in daily_rows if d["source"] == source and d["rows"] >= min_rows_per_day.get(source, 0))
    summary_rows.append({
        "source": source,
        "days_with_data": days_with_data,
        "total_rows": total_rows,
        "avg_rows_per_day": round(avg_rows, 2),
        "min_rows_day": min_rows,
        "max_rows_day": max_rows,
        "ok_days": ok_days,
        "dup_rate": round(dup_rate, 4),
        "null_article_id": round(null_rates.get("article_id", 0), 4),
        "null_title": round(null_rates.get("title", 0), 4),
        "null_url": round(null_rates.get("url", 0), 4),
        "min_published_at": str(min_pub) if min_pub is not None else None,
        "max_published_at": str(max_pub) if max_pub is not None else None,
        "files_scanned": total_files,
    })

summary_df = pd.DataFrame(summary_rows)
print("\nSUMMARY")
display(summary_df)

daily_df = pd.DataFrame(daily_rows)
if not daily_df.empty:
    pivot = daily_df.pivot_table(index="date", columns="source", values="rows", aggfunc="sum").fillna(0).astype(int)
    print("\nDAILY ROWS (last 14 days)")
    display(pivot.tail(14))
else:
    print("\nNo daily rows found in the scan window.")

news_root: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news
scan_days: 30 (cutoff: 2026-03-20)
min_rows_per_day: {'vnstock': 100, 'rss': 50, 'html': 50}

SUMMARY


,source,days_with_data,total_rows,avg_rows_per_day,min_rows_day,max_rows_day,ok_days,dup_rate,null_article_id,null_title,null_url,min_published_at,max_published_at,files_scanned
0,vnstock,2,8,4.0,3,5,0,0.1250,0.0,0.0,0.0,2026-02-03 10:14:19.317000+00:00,2026-04-10 15:30:00+00:00,2
1,rss,2,117,58.5,55,62,2,0.3333,0.0,0.0,0.0,2026-04-16 11:08:21+00:00,2026-04-17 17:05:00+00:00,2
2,html,2,199,99.5,99,100,2,0.4422,0.0,0.0,0.0,NaN,NaN,2



DAILY ROWS (last 14 days)


source,html,rss,vnstock
date,,,
2026-04-17,99,62,3
2026-04-18,100,55,5
